In [40]:
!pip install -Uq langchain langgraph langchain-community yfinance langchain-openai

In [43]:
from google.colab import userdata
import os
import yfinance as yf
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
import transformers
transformers.logging.set_verbosity_error()

from transformers import BertTokenizer, BertForSequenceClassification
import warnings
warnings.filterwarnings('ignore')

import torch

# Configure the API key in environment variables
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["HF_API_KEY"] = userdata.get('HF_API_KEY')

## Ingest and Classify Tool

In [44]:
from langchain_core.tools import tool as ingest_tool

@ingest_tool
def ingest_and_analyze_stock(query: str):
    """Extracts the stock ticker from the query, fetches news, and analyzes sentiment."""
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)
    llm = ChatOpenAI(model="gpt-4o-mini")

    # Use LLM to provide the stock ticker
    prompt = f"Extract the stock ticker symbol for the company mentioned in this query: '{query}'. Return ONLY the ticker symbol, nothing else."
    response = llm.invoke([HumanMessage(content=prompt)])
    ticker_symbol = response.content.strip()
    print(f"\nExtracted Ticker: {ticker_symbol}")

    # Fetch news for the ticker
    raw_news = yf.Search(ticker_symbol, max_results=5).news
    news_headlines = [item['title'] for item in raw_news if 'title' in item]

    print("\nNews Sentiment Analysis:")
    label_names = ["Negative", "Neutral", "Positive"]

    results = []
    # Use the transformer to classify the raw_news for sentiment analysis
    for headline in news_headlines:
        inputs = tokenizer(headline, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = model(**inputs)

        predicted_labels = torch.argmax(outputs.logits, dim=1)
        predicted_label = label_names[predicted_labels.item()]

        result_str = f"- {headline} -> Sentiment: {predicted_label}"
        print(result_str)
        results.append(result_str)

    return "\n".join(results)


## Re-routes to a new agent or tool when appropriate

In [46]:
from langchain_core.tools import tool as new_agent_tool

@new_agent_tool
def different_agent(query: str):
  """Takes questions not related to news and re-routes them to a different agent."""
  llm = ChatOpenAI(model="gpt-4o-mini")
  prompt = f"Recognize when another agent should take over when not news related from the query: '{query}'."
  other_agents = ""
  response = llm.invoke([HumanMessage(content=prompt)])
  return response.content


# Agent Tool calling

In [48]:


# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Provide the tool to the agent
tools = [ingest_and_analyze_stock, different_agent]

# Construct the Agent using LangGraph
agent_executor = create_react_agent(llm, tools)

# Invoke the agent asking about Nvidia stock
response = agent_executor.invoke({
    # Uncomment below to check ticker for query.
    "messages": [HumanMessage(content="What is the latest news and sentiment for Nvidia stock?")]
    # Testing tool call for none ticker question
    # "messages": [HumanMessage(content="What is the capitol of france?")]
})

print("\n--- Agent Response ---")
print(response["messages"][-1].content)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Extracted Ticker: NVDA

News Sentiment Analysis:
- Microsoft Needs Azure to Keep Growing Faster Than 40% When It Reports on Oct. 28 -> Sentiment: Positive
- Prediction: A $5,000 Investment in IonQ Could Be Worth This Much by 2028 -> Sentiment: Negative
- Dow Jones Futures Rise As Trump Claims Russia-Ukraine 'Energy Ceasefire'; ASML, Nvidia Chipmaker Taiwan Semi Earnings Loom -> Sentiment: Positive
- Amazon in talks to acquire AI startup Decart for $7 bln- WSJ -> Sentiment: Positive
- Not Nvidia. Not Broadcom. This Little-Known Chip Supplier Could Be the AI Boom's Best-Kept Secret. -> Sentiment: Neutral
- Forget Chasing Volume. Here's the Most Active Stock I'd Actually Hold for 5 Years. -> Sentiment: Neutral
- Micron Is Down 18% From Its High. History Suggests This Is What Happens Next. -> Sentiment: Neutral
- 3 Monthly Dividend Stocks Yielding Over 5% Worth Owning Now -> Sentiment: Neutral

--- Agent Response ---
Here's the latest news and sentiment analysis for Nvidia stock:

1. **Mi

In [49]:
# Checks what tool the llm called from query above.
for msg in response['messages']:
    print(f"\nMessage Type: {type(msg).__name__}")

    # If the AI decided to call a tool, it will be recorded in the 'tool_calls' attribute
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tool_call in msg.tool_calls:
            print(f"  -> Tool Selected: {tool_call['name']}")
            print(f"  -> Arguments Passed: {tool_call['args']}")

    # The actual output from the tool execution is stored in a ToolMessage
    elif type(msg).__name__ == 'ToolMessage':
        print(f"  -> Tool Output (truncated): {msg.content[:150]}...")


Message Type: HumanMessage

Message Type: AIMessage
  -> Tool Selected: ingest_and_analyze_stock
  -> Arguments Passed: {'query': 'Nvidia stock'}

Message Type: ToolMessage
  -> Tool Output (truncated): - Microsoft Needs Azure to Keep Growing Faster Than 40% When It Reports on Oct. 28 -> Sentiment: Positive
- Prediction: A $5,000 Investment in IonQ Co...

Message Type: AIMessage


In [ ]:
### Ignore this cell, used for reference. ###


# llm = ChatOpenAI(model="gpt-4o-mini")

# query = "Tell me about Nvidia?"

# prompt_template = ChatPromptTemplate.from_messages(
#     [
#         ("system", "You are a Financial Analyst who evaluates ticker {stock}"),
#         ("human", query)
#     ]
# )

# ticker = yf.Ticker(query)

# # get list of news
# raw_news = yf.Search(
#     query,
#     max_results=8,
#     news_count=8,
#     lists_count=8,
#     include_cb=True,
#     include_nav_links=False,
#     include_research=False,
#     include_cultural_assets=False,
#     enable_fuzzy_query=False,
#     recommended=8,
#     session=None,
#     timeout=30,
#     raise_errors=True
#     ).news

# # Extract headlines into a list
# news = [item['title'] for item in raw_news if 'title' in item]

# data = ticker.history(period="5d")

# print("Top News Headlines: \n", news)
# print("Historical Data: \n", data)



ERROR:yfinance:$TELL ME ABOUT NVIDIA?: No data found, symbol may be delisted


Top News Headlines: 
 []
Historical Data: 
 Empty DataFrame
Columns: [Open, High, Low, Close, Adj Close, Volume]
Index: []


In [ ]:
### Ignore this cell, used for reference. ###


# from langchain_core.messages import HumanMessage

# # Combine the headlines into a single string for the prompt
# news_text = "\n".join([f"- {headline}" for headline in news])

# prompt = (
#     "You are an expert financial analyst. Analyze the overall sentiment of the following news headlines "
#     "for {query}. Provide a brief summary of the main themes and state whether the overall "
#     "sentiment is Positive, Negative, or Neutral.\n\n"
#     f"Headlines:\n{news_text}"
# )

# print("Analyzing sentiment of the latest news...\n")

# # Invoke the LangChain LLM to get the sentiment analysis
# response = llm.invoke([HumanMessage(content=prompt)])
# print(response.content)

Analyzing sentiment of the latest news...

**Overall Sentiment Analysis and Summary**

The provided headlines generally reflect a mix of optimism and caution in the financial market, particularly centered on advancements in technology and investment strategies. 

**Main Themes:**
1. **Investment Potential**: The first headline indicates confidence in IonQ's future growth potential, suggesting an optimistic viewpoint for investors willing to take a long-term position.
2. **Market Movements**: The second headline regarding the Dow Jones futures hints at cautious optimism influenced by external geopolitical factors, which adds complexity to market sentiment.
3. **Acquisitions and Growth in AI**: Amazon's potential acquisition of Decart suggests a positive sentiment toward AI and technology innovation, reinforcing the bullish outlook around AI investments.
4. **Hidden Opportunities**: The mention of a lesser-known chip supplier being a great investment opportunity points to continued inves